# Explainability

Find out which features drive a model's predictions, overall and for one case, with permutation importance, SHAP and LIME.

This notebook comes from the [Explainability](https://pabl0aranda.github.io/Atlas-ML/testing/explainability/) page on Atlas ML, an interactive guide to machine learning. Run the cells from top to bottom.

In [ ]:
%pip install --quiet shap lime

## Example data

The code on the page assumes some data already exists. This cell creates it.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 569 tumors described by 30 measurements, each labeled malignant or benign.
data = load_breast_cancer()
X, y = data.data, data.target
feature_names, class_names = list(data.feature_names), list(data.target_names)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
print(f"Test accuracy: {model.score(X_test, y_test):.1%}")

## Code from the page

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance

# Global: how much the test score drops when each feature is shuffled.
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=0)
for i in result.importances_mean.argsort()[::-1][:5]:
    print(f"{feature_names[i]:<25} {result.importances_mean[i]:.3f}")

# SHAP: each feature's push on each prediction, for the positive class.
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)[:, :, 1]
shap.plots.bar(shap_values)        # global: average impact per feature
shap.plots.waterfall(shap_values[0])  # local: one prediction, feature by feature

# LIME: a simple model fitted around one prediction.
lime = LimeTabularExplainer(X_train, feature_names=feature_names, class_names=class_names, random_state=0)
print(lime.explain_instance(X_test[0], model.predict_proba, num_features=5).as_list())